# DAT494, Lab 4: Choose Your Own Adventure

Choose **one** of the following options and implement to the best of your ability. The details of implementation, training/testing data, and evaluation are left up to you.



### 2. Modernize Your GPT

Modernize your GPT-2 model with the following modifications/additions:

1. Implement a K-V Cache: Every attention layer should, after projecting the input sequence, check for cached keys/values to append to. After calculating attention, return these updated keys/values. You will also need to track a global list of key/value caches for all attention layers. The K-V cache should be used during inference (populated during pre-fill, and update each decoding step). You need not use it during pre-training.

2. Implement relative positional encodings with RoPE. Recall that you apply RoPE to the queries and keys only, and after you have performed the initial projections.

3. Replace the two-layer GPT-2 FFN with a three-layer (or two-layer with gating) SwiGLU FFN.

4. Add in grouped query attention (GQA): That is, share key and value projections across several query projections (i.e., query groups that share key/value projection). At a minimum, try to implement multi-query attention, where all query heads share a key/value projection (i.e., groups = 1).

4. **Optional:** Replace the FFNs with a Mixture of Experts (MoE). You may want to add an auxiliary loss term to improve load-balancing.

5. Train this model on some toy corpus. Then run the model on selected prompts in autoregressive mode such that the K-V cache is populated during prefill and then updated each decoding step, and confirm that you get plausible results.

In [ ]:
#Basic Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Basic PyTorch Libraries
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss

from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Part 1: Tokenizing and Chunking Text

- Get an appropriate GPT-2 byte-pair tokenizer (I suggest the HuggingFace version)
- Confirm that you can encode and decode sample text
- Get some kind of text training set of your choice, e.g., large books from Project Gutenberg, a Wikipedia dump, etc.
- Confirm that you can chunk this data into input/output sequences for next token ahead prediction
- Wrap into PyTorch Datasets and DataLoaders

In [ ]:
#import transformers from Hugging face to get GPT2 tokenizer

import transformers
from transformers import GPT2TokenizerFast

print(f"HuggingFace Transformers version: {transformers.__version__}")

Tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

HuggingFace Transformers version: 5.0.0


In [ ]:
MyInputText = "This class covers two or three graduate course portions"

#encoding to get token IDs
TokenID = Tokenizer.encode(MyInputText)
print(TokenID)

#decoding to confirm encoded text is the input text
DecodedText = Tokenizer.decode(TokenID)
print(DecodedText)



[1212, 1398, 8698, 734, 393, 1115, 10428, 1781, 16690]
This class covers two or three graduate course portions


In [ ]:
#Importing a book as a file  from Project Gutenberg for training data using gutenberg URL
#obtained URL by google search and colab autofill
import urllib.request

GutenbergURL = "https://www.gutenberg.org/files/1342/1342-0.txt"
urllib.request.urlretrieve(GutenbergURL, "PrideAndPrejudice.txt")

with open("PrideAndPrejudice.txt", "r", encoding="utf-8") as f:
    PrideAndPrejudice = f.read()
print(f"Downloaded {len(PrideAndPrejudice):,} characters from {GutenbergURL}")

Downloaded 728,846 characters from https://www.gutenberg.org/files/1342/1342-0.txt


In [ ]:
EntirebookTokens = Tokenizer.encode(PrideAndPrejudice)
#Creating a tensor to feed tokens into models for training
EntirebookTokens = torch.tensor(EntirebookTokens)
print(f"EntirebookTokens shape: {EntirebookTokens.shape}")

NumberofTokens = len(EntirebookTokens)
print(f"The book has {NumberofTokens:,} tokens")

#Creating Input and output seq using chunking.
#Match the model's reduced context length to avoid indexing errors
ChunkSize = 512
NumberofChunks = NumberofTokens // ChunkSize
print(f"The book has {NumberofChunks:,} chunks")

input_seq = EntirebookTokens[:NumberofChunks*ChunkSize].view(-1, ChunkSize)
ouput_seq = EntirebookTokens[1:NumberofChunks*ChunkSize+1].view(-1, ChunkSize)

print(f"input_seq shape: {input_seq.shape}")
print(f"ouput_seq shape: {ouput_seq.shape}")

Token indices sequence length is longer than the specified maximum sequence length for this model (191673 > 1024). Running this sequence through the model will result in indexing errors


EntirebookTokens shape: torch.Size([191673])
The book has 191,673 tokens
The book has 374 chunks
input_seq shape: torch.Size([374, 512])
ouput_seq shape: torch.Size([374, 512])


In [ ]:
#creating a token dataset using the input_seq and output_seq created from the book tokens and split that into training and validation data sets
#using 80-20 training, validation split
TokenDataset = torch.utils.data.TensorDataset(input_seq, ouput_seq)
n_train = int(0.8  * len(TokenDataset))
n_val = len(TokenDataset) - n_train
train_set, val_set = torch.utils.data.random_split(TokenDataset, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False)

#input tokens and the expected prediction
x_batch, y_batch = next(iter(train_loader))
print(f"x_batch shape: {x_batch.shape}")
print(f"y_batch shape: {y_batch.shape}")

x_batch shape: torch.Size([8, 512])
y_batch shape: torch.Size([8, 512])


## Part 2: Embedding Text

- Confirm that you can perform the following embeddings into a vector space of dimension `d_model`

1. Token embedding
2. Positional embedding, allow up to `context_length` positions

- Then, confirm you can sum the two for an overall sequence embedding

In [ ]:
#parameters for vocab size, model dimensions and context length
Vocab_size = Tokenizer.vocab_size
Context_length = 1024
d_model = 768

print(f"Vocab_size: {Vocab_size}")

#Token embedding & Positional embedding
Token_embedding = nn.Embedding(Vocab_size, d_model)
sample_token_embedding = Token_embedding(torch.tensor([1212, 1398, 8698, 734, 393, 1115, 10428, 1781, 16690]))
print(f"Token embedding shape: {sample_token_embedding.shape}")

#Create positional table for context length
Position_table = nn.Embedding(Context_length, d_model)
sample_position_embedding = Position_table(torch.arange(len(TokenID))) # Fix: Pass the length of TokenID to arange
print(f"Positional embedding shape: {sample_position_embedding.shape}")


#sequence embedding
sample_sequence_embedding = sample_token_embedding + sample_position_embedding
print(f"Sequence embedding shape: {sample_sequence_embedding.shape}")


Vocab_size: 50257
Token embedding shape: torch.Size([9, 768])
Positional embedding shape: torch.Size([9, 768])
Sequence embedding shape: torch.Size([9, 768])


## Part 3:Dummy Modern GPT Model

- Create a dummy GPT Model class, as well as the following necessary dummy classes:

1. Dummy Multi-Headed Causal Attention
2. Dummy Layer Normalization
3. Dummy Feed-Forward Block
4. Dummy Overall Transformer Block

- Do not forget to include Dropout layers for a true GPT-2 clone

- Use the above dummy classes, along with your text preparation and embedding steps, to confirm that the dummy model can process arbitrary input sequences (of length up to the context window)


In [ ]:
#nn.LayerNorm already performs layer normalization. creating a class using model param
class DummyLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x)

In [ ]:
class DummySwiGLUFeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        # In SwiGLU, we expand to 8/3 * d_model instead of 4 * d_model above
        # to account for 3 linear layers instead of 2
        intermediate_dim = int(8/3 * d_model)

        self.linear1 = nn.Linear(d_model, intermediate_dim, bias=False )
        self.linear2 = nn.Linear(d_model, intermediate_dim, bias=False )
        self.linear3 = nn.Linear(intermediate_dim, d_model, bias=False) # restore to model dim

    def forward(self, x):
        # Using pytorch provided SiLU to obtain SwiGLU(x) = (SiLU(xW1) * xW2) * W3
        gate = torch.nn.functional.silu(self.linear1(x))
        x = gate * self.linear2(x)
        return self.linear3(x)

In [ ]:
#implementing RoPE here
"""
RoPE implements rotational position embedding so that the attention score is proportional
to the relative distance bewteen tokens. This allows models to extrapolate context windows
So a model trained on 8K tokens can be fined tuned to handle 256K token context window

"""

def rope(q, k, device):
    seq_len = q.shape[1]
    head_dim = q.shape[-1]

    # Generate frequencies and create interpolation
    inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float().to(device) / head_dim))
    t = torch.arange(seq_len, device=device).type_as(inv_freq)
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)

    cos = emb.cos().view(1, seq_len, 1, head_dim)
    sin = emb.sin().view(1, seq_len, 1, head_dim)

    # Rotate the vectors
    def rotate_half(x):
        x1, x2 = x[..., :head_dim//2], x[..., head_dim//2:]
        return torch.cat((-x2, x1), dim=-1)

    xq_out = (q * cos) + (rotate_half(q) * sin)
    xk_out = (k * cos) + (rotate_half(k) * sin)
    return xq_out, xk_out

In [ ]:
"""
Attention calculation uses QKV, in particular K transpose.  Attention(Q,K,V) = softmax(Q*K^T/sqrt(d_k))*V
So compute Q, K, V using linear layers. Then plug in Q,K,V into the above expression for Attention calculation
1/sqrt(d_k) is the scaling factor. Keeping in mind, in the decoder, masked attention is what we calculate, we
apply the appropriate masking to the attention matrix. Number of heads is a parameter that allows experimenting
with different number of heads in the multi-head attention mechanism.

We implement KV caching here. Since for every token i+1 we need to compute KV for tokens 0 to i, we waste a lot of
compute redoing the calculations. So we storing the previously calculated KV values for previous tokens will help
reduce compute. So we implement a KV cache to do just that.

In a typical Multi-head attention we calculate KV per head. Since we are using KV cache here,
KV cache grows with sequence length. To limit the KV cache, we will implement group query attention.
In this method, we share KV for several Query heads, thus saving KV cache space. we introduce a new parameter
called n_kv_heads
"""


class DummyGroupQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, context_length):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.kv_group_size = n_heads // n_kv_heads
        # Ensure kv_dim is an integer, as it's used for nn.Linear's out_features
        self.kv_dim = int(d_model / n_heads * n_kv_heads)
        self.d_k = d_model // n_heads


        self.Q = nn.Linear(d_model, d_model, bias=False)
        self.K = nn.Linear(d_model, self.kv_dim, bias=False)
        self.V = nn.Linear(d_model, self.kv_dim, bias=False)
        self.O = nn.Linear(d_model, d_model, bias=False)

        self.scale = 1/np.sqrt(self.d_k)

        # Registering the mask as a buffer so it automatically moves with the model to the correct device
        self.register_buffer('mask', torch.tril(torch.ones(context_length, context_length)))

    def forward(self, x, cache=None):
        B, S, C = x.shape

        # Project
        Q = self.Q(x)
        K = self.K(x)
        V = self.V(x)

        # realign the tensors for matrix multiplication to calculate attention scores and to
        # slice the linear layer output per head for Q, but for KV only the # of kv_heads
        Q = Q.view(B, S, self.n_heads, self.d_k)
        K = K.view(B, S, self.n_kv_heads, self.d_k)
        V = V.view(B, S, self.n_kv_heads, self.d_k)

        # Apply RoPE after initial projections
        Q,K = rope(Q,K, x.device)

        # K-V Caching implementation. Cache is set to None during training
        if cache is not None:
            prev_k, prev_v = cache
            K = torch.cat([prev_k, K], dim=1) # Add to cache along sequence dimension
            V = torch.cat([prev_v, V], dim=1)
        new_cache = (K, V)

        # We repeat KV heads to match the number of Query heads for dot product calculation
        K_rep = K.repeat_interleave(self.kv_group_size, dim=2)
        V_rep = V.repeat_interleave(self.kv_group_size, dim=2)

        # Caculate attention score based on the formula above
        Q = Q.transpose(1, 2)
        K_rep = K_rep.transpose(1, 2)
        V_rep = V_rep.transpose(1, 2)
        scores = torch.matmul(Q, K_rep.transpose(-2, -1)) * self.scale

        # Apply mask (only needed during training/pre-fill)
        curr_mask = self.mask[:S, :K.shape[1]]
        scores = scores.masked_fill(curr_mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V_rep)

        out = out.transpose(1, 2).contiguous().view(B, S, C)
        return self.O(out), new_cache

In [ ]:
# Dummy Modern Transformer block to put together the components defined above
# the transformer is a sequence of layernorm followed by Group Query attention
# and then a layernorm followed by SwiGLU feedforward. Making residual connections as needed

class DummyModernTransformerBlock(nn.Module):
  def __init__(self, d_model, n_heads, n_kv_heads, context_length):
    super().__init__()

    # Pass context_length to the attention layer
    self.attention = DummyGroupQueryAttention(d_model, n_heads, n_kv_heads, context_length)
    self.norm1 = DummyLayerNorm(d_model)
    self.feedforward = DummySwiGLUFeedForward(d_model)
    self.norm2 = DummyLayerNorm(d_model)
    self.dropout = nn.Dropout(0.1)

  def forward(self, x, cache=None):
    # Attention Block
    residual = x
    x_normed = self.norm1(x)
    attn_output, new_cache = self.attention(x_normed, cache=cache)
    x = residual + self.dropout(attn_output) # Add residual connection

    # SwiGLU Feed-forward Block
    residual = x
    x_normed = self.norm2(x)
    ff_output = self.feedforward(x_normed)
    x = residual + self.dropout(ff_output)
    return x, new_cache # Transformer block returns x and the cache from its attention layer

## Part 4: Fill in All Necessary Blocks

- Fill in all your dummy layers/blocks above, and deliver a final GPT model that is ready for training


In [ ]:
class DummyModernGPT(nn.Module):
  def __init__(self, vocab_size, context_length, d_model, n_heads, n_kv_heads, n_blocks):
    super().__init__()
    self.d_model = d_model
    self.n_heads = n_heads
    self.n_kv_heads = n_kv_heads
    self.n_blocks = n_blocks
    self.vocab_size = vocab_size
    self.context_length = context_length

    self.token_embedding = nn.Embedding(vocab_size, d_model)
    self.position_embedding = nn.Embedding(context_length, d_model)
    self.emb_dropout = nn.Dropout(0.1)

    # Create transformer blocks and an output head using ModuleList for proper iteration
    self.blocks = nn.ModuleList([DummyModernTransformerBlock(d_model, n_heads, n_kv_heads, context_length) for _ in range(n_blocks)])
    self.layer_norm = DummyLayerNorm(d_model)
    self.output_head = nn.Linear(d_model, vocab_size)

  def forward(self, x, caches=None):
    # Check
    Batch, Chunk_length = x.shape
    if Chunk_length > self.context_length:
        raise ValueError(f"Input sequence length {Chunk_length} exceeds context length {self.context_length}")

    # Apply token and position embeddings first
    x_token_emb = self.token_embedding(x)
    position_ids = torch.arange(Chunk_length, device=x.device)
    position_ids = position_ids.expand(Batch, Chunk_length)
    x_pos_emb = self.position_embedding(position_ids)
    x = x_token_emb + x_pos_emb
    x = self.emb_dropout(x)

    new_caches = []
    for i, block in enumerate(self.blocks):
      layer_cache = None # Initialize layer_cache
      if isinstance(caches, (list, tuple)) and i < len(caches):
         layer_cache = caches[i]
      x, cache = block(x, cache=layer_cache)
      new_caches.append(cache)

    # Apply final layer norm and output head
    x = self.layer_norm(x)
    logits = self.output_head(x)
    return logits, new_caches

In [ ]:
vocab_size = Tokenizer.vocab_size
context_length = 512
d_model = 128
n_heads = 4
n_kv_heads = 2
n_blocks = 12
short_seq = torch.randint(0, vocab_size, (1, 10))
medium_seq = torch.randint(0, vocab_size, (1, 256))
long_seq = torch.randint(0, vocab_size, (1, context_length))
dummy_model = DummyModernGPT(vocab_size, context_length, d_model, n_heads, n_kv_heads, n_blocks)
short_logits, _ = dummy_model(short_seq) # Capture both logits and new_caches
print(f"short seq shape: {short_seq.shape}")
print(f"short logits shape: {short_logits.shape}")
medium_logits, _ = dummy_model(medium_seq) # Capture both logits and new_caches
print(f"medium seq shape: {medium_seq.shape}")
print(f"medium logits shape: {medium_logits.shape}")
long_logits, _ = dummy_model(long_seq) # Capture both logits and new_caches
print(f"long seq shape: {long_seq.shape}")
print(f"long logits shape: {long_logits.shape}")

# using the real sample input text created above to check the model
# Convert TokenID list to a tensor before passing it to the model
TokenID_tensor = torch.tensor(TokenID, dtype=torch.long).unsqueeze(0)
sample_logits, _ = dummy_model(TokenID_tensor) # Capture both logits and new_caches
print(f"sample seq shape: {TokenID_tensor.shape}")
print(f"sample logits shape: {sample_logits.shape}")

short seq shape: torch.Size([1, 10])
short logits shape: torch.Size([1, 10, 50257])
medium seq shape: torch.Size([1, 256])
medium logits shape: torch.Size([1, 256, 50257])
long seq shape: torch.Size([1, 512])
long logits shape: torch.Size([1, 512, 50257])
sample seq shape: torch.Size([1, 9])
sample logits shape: torch.Size([1, 9, 50257])


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, loss_val):
        if self.best_loss is None:
            self.best_loss = loss_val
        elif loss_val > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = loss_val
            self.counter = 0

## Part 5: Next Token Ahead Pre-Training

(Pre-) train the model on your chosen corpus using next token ahead prediction. Use reasonable structural parameters for the model that allow you to train over a sensible amount of time. You may want to use a GPU (e.g., Google Colab) for training, but this is not required.

In [ ]:
model = DummyModernGPT(vocab_size, context_length, d_model, n_heads, n_kv_heads, n_blocks).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)  # Better than adam due to how weight decay is directly applied to parameters
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)  # Halves learning rate if there is no improvement over 2 epochs
stopper = EarlyStopping(patience=5)

num_epochs = 50
loss_func = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    # Training Loss Calculation
    model.train()
    train_losses = []
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        logits, _ = model(X) # Unpack the logits and ignore new_caches for training
        loss = loss_func(logits.view(-1, model.vocab_size), y.view(-1))
        # Logits need to be reshaped so that the output tokens can be aligned
        # to target labels

        optimizer.zero_grad()
        loss.backward()

        # Implementing gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_losses.append(loss.item())
    # Validation Loss Calculation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for X, Y in val_loader:
            X, Y = X.to(device), Y.to(device)
            logits, _ = model(X) # Unpack the logits and ignore new_caches for validation
            v_loss = loss_func(logits.view(-1, model.vocab_size), Y.view(-1))
            val_losses.append(v_loss.item())

    avg_train_loss = sum(train_losses) / len(train_losses)
    avg_val_loss = sum(val_losses) / len(val_losses)

    print(f"Epoch: {epoch}, Average Training Loss: {avg_train_loss:.4f}, Average Validation Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)
    stopper(avg_val_loss)

    if stopper.early_stop:
        print(f"Early stopping triggered at epoch {epoch}")
        break

Epoch: 0, Average Training Loss: 7.9200, Average Validation Loss: 6.1002
Epoch: 1, Average Training Loss: 5.7745, Average Validation Loss: 5.5706
Epoch: 2, Average Training Loss: 5.2487, Average Validation Loss: 5.1868
Epoch: 3, Average Training Loss: 4.8581, Average Validation Loss: 4.9359
Epoch: 4, Average Training Loss: 4.5501, Average Validation Loss: 4.7732
Epoch: 5, Average Training Loss: 4.2924, Average Validation Loss: 4.6752
Epoch: 6, Average Training Loss: 4.0698, Average Validation Loss: 4.6430
Epoch: 7, Average Training Loss: 3.8653, Average Validation Loss: 4.6091
Epoch: 8, Average Training Loss: 3.6702, Average Validation Loss: 4.6136
Epoch: 9, Average Training Loss: 3.4943, Average Validation Loss: 4.6539
Epoch: 10, Average Training Loss: 3.3111, Average Validation Loss: 4.6805
Epoch: 11, Average Training Loss: 3.1172, Average Validation Loss: 4.7272
Epoch: 12, Average Training Loss: 2.9865, Average Validation Loss: 4.7770
Early stopping triggered at epoch 12


## Part 6: Autoregressive Text Generation

- Implement a function or method to autoregressively generate text with your model, starting from some seed text
- Allow greedy or stochastic decoding, and specify temperature and top-k (possible None) parameters for decoding
- Play around with generating text, include some examples you find interesting

In [ ]:
"""
Since we implemented KV caching, we enable KV caching in the model and run the model with sample prompt
We prefill the caches with KV calculations for all tokens. Then during decode we only calculate Q,K,V for
the last token

"""
def text_gen_with_cache(model, tokenizer, seed_text, max_new_tokens=50, context_length=512, temperature=1.0, top_k=None, greedy=False):
    model.eval()
    input_tokens = tokenizer.encode(seed_text)
    input_tensor = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

    #prefill here. calculate KV values for all tokens and store it in cache
    logits, caches = model(input_tensor)
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    all_tokens = torch.cat([input_tensor, next_token], dim=1)

    for i in range(max_new_tokens):

        with torch.no_grad():
            # now model calculates Q,K,V only for the last token
            logits, caches = model(all_tokens[:,-1:], caches=caches)
            logits = logits[:, -1, :] / temperature

            if greedy:
                # Greedy decoding picks the token with the maximum value (hence the use of argmax)
                next_token = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                # Applies softmax function so that future tokens can be sampled from probability distribution
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')

                # Sampling and creation of the probability distribution
                probs = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

            all_tokens = torch.cat((all_tokens, next_token), dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(all_tokens.squeeze().tolist())

print("Greedy Decoding:\n")
print(text_gen_with_cache(model, Tokenizer, "The surest way to disappoint him will be", greedy=True))
print("\n")

print("Stochastic Decoding:\n")
print(text_gen_with_cache(model, Tokenizer, "The surest way to disappoint him will be", top_k=50, temperature=0.8))

Greedy Decoding:

The surest way to disappoint him will be
the; and which had been concerned; and which had been concerned; and which had been concerned; and which had been concerned; and which had been concerned; and which had been concerned; and which had been concerned; and which had been concerned;


Stochastic Decoding:

The surest way to disappoint him will be
her; and which had beenat; been formerly, and which had received, of her father, and which of their following
acquness and which had been mostension; and; and, andither; and the country, and which became;


## Optional Part 7: Fine-Tune for Classification

Optionally, you may replace the final language modeling head of your model with a classification head, and fine-tune the model for text classification. A simple example is sentiment analysis on the IMDB Database, but many other classification datasets are available as well.

## Optional Part 8: Supervised Fine-Tuning on Instruction Data

Also optionally, you may follow our work in class to perform supervised fine-tuning on an instruction dataset, such as the Alpaca Instruction Set (https://crfm.stanford.edu/2023/03/13/alpaca.html).